# 11장. LLM과 함께 분석 질문을 다듬기

이 노트북은 외부 LLM을 직접 호출하지 않습니다. Chapter 05에서 만든 processed 데이터를 사용해 **원본 행 없이 안전한 구조 Context, 검증 가능한 Prompt Template, 사람 검토 Checklist, 실제 사용 전에는 `not_executed`로 표시되는 Log Template**을 만듭니다.


## 📌 이번 장 한눈에 보기 (초보자용 요약)
- **목표:** LLM에게 데이터를 통째로 주지 않고, **안전하게 줄인 정보(Safe Context)**만 주고 → 답을 **증거(Evidence)로 검증**하는 연습
- **흐름:** processed 데이터 확인 → 구조만 요약 → 민감정보 점검 → Safe Context 만들기 → 자동 검사 → 사람이 확인 → Prompt 작성 → 답변 검증 → 사용 기록
- **핵심 한 줄:** 좋은 Prompt가 정답을 보장하지 않는다. 검증하는 사람이 중요하다.
- **주의:** 이번 실습 코드는 외부 LLM을 직접 부르지 않는다.

## 학습 목표

- raw 데이터로 조용히 fallback하지 않고 processed 입력을 확인합니다.
- 실제 값 예시 없이 데이터 구조를 요약합니다.
- 직접 민감정보로 추정되는 컬럼명은 Safe Context에서 기본 제외합니다.
- 식별자 컬럼은 관계 설명을 위해 이름만 사용할 수 있지만 원본 값은 공유하지 않습니다.
- Prompt에 목적·구조·요청·제약·출력·검증 조건을 함께 작성합니다.
- Chapter 09/10과 같은 예측 시점·누수·선택/테스트 기준을 LLM 요청에도 적용합니다.
- 외부 문서의 지시문은 명령이 아니라 `untrusted data`로 취급합니다.
- 실제 LLM을 사용하지 않은 빈 Log를 사용 증거로 오해하지 않습니다.


## 1. 프로젝트 루트와 processed 입력 설정


In [1]:
from pathlib import Path
import sys

def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')

PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]
missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        'Chapter 11은 processed 데이터에서 시작합니다. 먼저 python scripts/preprocess_data.py를 실행하세요. 누락: '
        + ', '.join(str(path) for path in missing_files)
    )

print('프로젝트 루트:', PROJECT_ROOT)
print('processed 입력:', PROCESSED_DIR)
print('결과 폴더:', REPORT_DIR)


프로젝트 루트: C:\dev\llm-data-analysis-course
processed 입력: C:\dev\llm-data-analysis-course\data\processed
결과 폴더: C:\dev\llm-data-analysis-course\reports


### STEP 0 정리
- **결과 관찰:** processed 4개 파일(customers, products, orders, order_items)이 모두 있었고 오류 없이 통과했다.
- **나의 해석과 판단:** raw로 자동 대체하지 않는 이유는, 정제 전 데이터가 모르는 사이에 기준이 되는 것을 막기 위해서다.
- **사용 여부:** 사용
- **Evidence:** `required_files` 확인 코드가 FileNotFoundError 없이 통과
- **한계와 추가 확인 사항:** 파일이 있다는 것만 확인했고, 내용이 올바른지는 다음 단계에서 본다.

## STEP 1. LLM 사용 전 분석 질문과 역할 정의

### 분석 질문
completed 주문 기준 카테고리별 주문 상세 금액(`line_total`) 비중은 어떻게 다른가?

### 현재 데이터로 계산 가능한가?
- **필요한 지표:** 카테고리별 `line_total` 합계 ÷ 전체 합계
- **분석 단위:** 카테고리 (주문 상세 `order_items` 행을 집계)
- **필요한 데이터:** `order_items`(금액), `orders`(주문 상태), `products`(카테고리)
- **판단:** 필요한 컬럼이 모두 있어 계산 가능하다.
- **한계:** 이 세 가지는 집계 전에 확인해야 한다.
  - `category`에 띄어쓰기만 다른 표기가 있다 (`전자기기`/`전자 기기`, `생활용품`/`생활 용품`, `뷰티`/`뷰 티`).
  - `order_items` 26행의 `product_id`가 `products`에 없다.
  - `order_items` 2행의 `order_id`가 `orders`에 없다.

### 역할 구분
| 구분 | 맡을 일 |
|---|---|
| **LLM** | 필요한 집계 코드와 검증 체크리스트 초안 제안 |
| **사람(나)** | 실제 컬럼명, 병합 관계, 합계 일치, 해석 범위 확인과 최종 판단 |

### 검증 방법
병합 행 수, 미매칭 행 수, source total과 group total 일치를 STEP 8에서 확인한다.

## 2. Safe Context와 Prompt 자료 생성

공통 모듈은 processed 4개 파일이 모두 없으면 중단합니다. raw 데이터로 자동 fallback하지 않습니다. 이 단계에서도 외부 LLM API는 호출하지 않습니다.


In [2]:
from src.llm_prompt_analysis import run_llm_prompt_analysis

result = run_llm_prompt_analysis(
    processed_dir=PROCESSED_DIR,
    raw_dir=RAW_DIR,
    report_dir=REPORT_DIR,
)

print('사용 데이터:', result['source_type'])
print('외부 LLM 호출: 없음')


사용 데이터: processed
외부 LLM 호출: 없음


## 3. 데이터셋·컬럼 구조와 민감정보 검토


In [ ]:
dataset_summary = result['dataset_summary']
column_summary = result['column_summary']
sensitive_review = result['sensitive_review']

display(dataset_summary)
display(column_summary)
display(sensitive_review)


### 결과 요약
| 데이터셋 | 행 | 컬럼 | 결측 | 중복행 | 식별자 컬럼 |
|---|---|---|---|---|---|
| customers | 150 | 6 | 2 | 0 | 1 |
| products | 96 | 4 | 0 | 0 | 1 |
| orders | 300 | 7 | 9 | 0 | 2 |
| order_items | 752 | 6 | 0 | 0 | 3 |

- 네 데이터셋 모두 `source_type = processed`다.
- 결측은 customers 2개, orders 9개다. 중복 행은 없다.

`column_summary`에는 실제 값 예시가 없습니다. `share_raw_values`는 항상 `no`이며, 직접 민감정보로 추정되는 컬럼명은 `do_not_share_name_by_default`, ID 계열은 `share_name_only_no_values`로 표시합니다. `low_cardinality_review=True`는 값 자체가 노출됐다는 뜻이 아니라 소수 범주 집계 시 재식별 위험을 사람이 다시 보라는 신호입니다.


### ⚠️ 주의가 필요한 컬럼
| 구분 | 컬럼 | 이유 | 대응 |
|---|---|---|---|
| 이름 패턴 | customers.name, products.product_name | 이름처럼 보이는 컬럼 | 컬럼명과 값 모두 기본 공유 금지 |
| 소수 범주 | customers.gender, orders.payment_method, orders.order_status | 값 종류가 적어 재식별 위험 | 값과 함께 집계할 때 검토 |
| 식별자 | customer_id, product_id, order_id, order_item_id | ID | 컬럼명만 검토, ID 값은 공유 금지 |

모든 항목의 `human_decision`이 `review_required`다. 사람이 직접 결정해야 한다.

### STEP 2 성공 기준 확인
- `share_raw_values`는 23개 컬럼 **모두 `no`**다. 실제 값 예시는 들어 있지 않다.
- `identifier` 7개는 `share_name_only_no_values`, 이름 패턴 2개는 `do_not_share_name_by_default`로 표시됐다.
- `low_cardinality_review = True`는 3개(gender, payment_method, order_status)다. 값이 노출됐다는 뜻이 아니라 집계 시 재식별 위험을 다시 보라는 신호다.
- **사용 여부:** 사용 / **한계:** 컬럼 수와 종류만 확인했고, 조직 정책 검토는 아직 하지 않았다.

## 4. 외부 LLM용 Safe Context 확인


In [ ]:
safe_context_text = result['safe_context_text']
context_validation = result['context_validation']

print(safe_context_text)
display(context_validation)


### Safe Context 내용 관찰
- 데이터셋 4개의 행·열·결측·중복만 담겨 있고, 실제 행이나 값은 없다.
- `customers.name`, `products.product_name`은 컬럼명도 기본 제외됐다.
- ID 컬럼은 이름과 고유값 수만 있고 값은 없다.
- 분석에 필요한 `products.category`(고유값 10개), `order_items.line_total`, `orders.order_status`는 Context에 들어 있다.
- **관찰 하나:** `category` 고유값이 10개인데, STEP 1에서 본 표기 불일치(띄어쓰기만 다른 값)가 이 10개에 섞여 있을 수 있다. Safe Context만으로는 알 수 없다.

### 자동 Validation 해석
- 5개 항목 모두 PASS다.
- **자동 PASS는 외부 LLM 제공 승인이 아니다.** 기술적 최소 조건일 뿐이고, 실제 제공 전에는 조직 정책과 사람 검토가 필요하다.

Safe Context는 **자동 승인 자료가 아닙니다.** 다음을 사람이 다시 확인합니다.

- 조직에서 허용한 LLM 도구와 계정인가?
- 컬럼명 자체가 내부 업무나 민감 속성을 드러내지 않는가?
- 소수 집단·희귀 범주 집계가 개인을 추정하게 하지 않는가?
- 오류 메시지·파일 경로·내부 URL·Secret이 없는가?
- 외부 웹/PDF/이메일 안의 지시문을 실행 명령으로 따르지 않는가?


## 5. Prompt Template 검토


In [3]:
prompt_templates = result['prompt_templates']
display(
    prompt_templates[[
        'step', 'purpose', 'prompt_version',
        'context_rule', 'human_review_required', 'validation_point'
    ]]
)


,step,purpose,prompt_version,context_rule,human_review_required,validation_point
0,분석 질문 생성,현재 데이터로 답할 수 있는 EDA 질문 후보 만들기,2.0,safe_context only; no raw rows/secrets/direct ...,True,질문·지표·분석 단위가 실제 구조와 completed 범위에 맞는지 확인
1,전처리 계획,삭제보다 처리 선택지와 검증 기준을 먼저 정리하기,2.0,safe_context only; no raw rows/secrets/direct ...,True,손실 행·변환 실패·PK/FK·원본 보존 여부를 사람이 결정했는지 확인
2,시각화 설계,질문에 맞는 그래프와 축·범위·해석 한계 선택하기,2.0,safe_context only; no raw rows/secrets/direct ...,True,그래프와 저장 집계의 범위·축·단위·개인정보가 일치하는지 확인
3,회귀 코드 검토,Chapter 09 기준의 예측 시점·누수·시간 평가를 검토하기,2.0,safe_context only; no raw rows/secrets/direct ...,True,Chapter 09의 prediction-time·train-only selecti...
4,분류 코드 검토,Chapter 10 기준의 취소 분류 계약을 검토하기,2.0,safe_context only; no raw rows/secrets/direct ...,True,Chapter 10의 target·feature contract·validation...
5,결과 해석,관찰·가설·추가 검증·한계를 분리하기,2.0,safe_context only; no raw rows/secrets/direct ...,True,관찰과 인과 가설이 분리되고 데이터 범위를 넘어선 단정이 없는지 확인
6,외부 문서 검토,Prompt Injection이 포함될 수 있는 외부 콘텐츠를 안전하게 다루기,2.0,safe_context only; no raw rows/secrets/direct ...,True,외부 콘텐츠의 지시문이 실행 명령으로 승격되지 않았는지 확인


In [4]:
for selected_step in ['회귀 코드 검토', '분류 코드 검토', '외부 문서 검토']:
    print('\n' + '=' * 70)
    print(selected_step)
    print('=' * 70)
    prompt = prompt_templates.loc[
        prompt_templates['step'] == selected_step, 'prompt'
    ].iloc[0]
    print(prompt)



회귀 코드 검토
역할: 머신러닝 코드 리뷰어
목표: 주문 상세가 완성되기 전 주문 총금액(order_total)을 예측하는 교육용 회귀 실습을 검토합니다.
예측 시점: 주문 상세 기반 target 재료를 아직 사용할 수 없는 시점입니다.
금지 후보: line_total, quantity, unit_price, item_count, total_quantity, avg_unit_price, order_total, order_status, order_id, customer_id.
허용 후보 예: payment_method, 주문 월/요일, 고객 age/gender/city 등 예측 시점에 존재한다고 확인된 값.
요청:
1. 모든 feature의 예측 시점 가용성을 감사하세요.
2. target 계산 재료·사후정보·식별자를 누수로 표시하세요.
3. 날짜 순서로 train/test를 분리하고 동일 날짜가 양쪽에 섞이지 않게 하세요.
4. train 내부 TimeSeriesSplit으로 Dummy/Linear/RandomForest 후보를 비교하세요.
5. 선택 모델을 고정한 뒤 final test를 한 번 평가하세요.
6. MAE, RMSE, R²와 Dummy 개선 여부를 함께 보고하세요.
제약: test 결과로 모델을 다시 선택하지 마세요.
검증 요청: prediction time, leakage, split, baseline, selection data, final test 사용을 표로 확인하세요.

분류 코드 검토
역할: 머신러닝 코드 리뷰어
목표: 주문 생성 직후 정보로 주문 취소 위험을 예측합니다.
타깃: completed=0, cancelled=1; refunded와 기타 상태는 제외합니다.
예측 시점 가정: item_count/total_quantity/order_amount는 주문 생성 시 상품 구성과 금액이 확정된 교육용 가정 아래에서만 허용합니다.
금지 입력: order_status, is_cancelled, order_id, customer_id,

### STEP 4. 나의 Prompt Contract

**나쁜 Prompt**
> 이 데이터를 분석해 주세요.

→ 목적, 범위, 계산 기준, 검증 조건이 없어서 그럴듯하지만 확인할 수 없는 답이 나온다.

**더 나은 Prompt (STEP 1 질문 기준)**
| 요소 | 내용 |
|---|---|
| 역할 | Python 데이터 분석 검토자 |
| 목적 | completed 주문 기준 카테고리별 주문 상세 금액 비중 계산 |
| 승인된 Context | 사람이 검토한 Safe Context (실제 행·ID 값 없음) |
| 요청 | pandas 집계 코드, 설명, 검증 체크리스트 |
| 제약 | 없는 컬럼 만들지 않기, 개인정보 요구하지 않기 |
| 계산 기준 | `line_total = quantity × unit_price`, `order_status = completed`만 포함 |
| 출력 형식 | 코드 → 설명 → 검증 체크리스트 순서 |
| 검증 조건 | 병합 행 수, 미매칭 행 수, source total과 group total 일치 |

**Prompt 설계 이유**
- Context는 Safe Context로 한정해서 원본 데이터가 외부로 나가지 않게 한다.
- `category` 표기 불일치와 미매칭 행은 LLM이 모를 수 있어서, 검증 조건에 미매칭 확인을 넣는다.

- **사용 여부:** 작성만 함 (실제 LLM에는 아직 넣지 않았다)
- **한계:** `line_total`이 정말 `quantity × unit_price`와 같은지는 아직 확인하지 않았다. STEP 8에서 검증한다.

회귀 Prompt는 Chapter 09의 **예측 시점 → target 재료 누수 제외 → 날짜 순서 분할 → train 내부 TimeSeriesSplit 선택 → frozen final test** 흐름을 따릅니다. 분류 Prompt는 Chapter 10의 **completed/cancelled 범위 → Feature Contract → validation model/threshold 선택 → frozen test** 흐름을 따릅니다.


## 6. LLM 답변 검증 Checklist


In [ ]:
checklist = result['checklist']
display(checklist)


검증은 네 층으로 봅니다: **입력·보안 → 코드/수치 → 모델 → 해석**. 체크박스가 비어 있는 상태는 검증 완료를 의미하지 않습니다.


## STEP 8. LLM 답변을 Evidence Matrix로 검증
STEP 1의 질문(completed 주문 기준 카테고리별 금액 비중)에 대한 LLM 제안을 **그럴듯함이 아니라 실제 데이터로** 검증한다.

In [5]:
import pandas as pd

products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('컬럼 존재:', 'category' in products, 'line_total' in items, 'order_status' in orders)
print('금액 결측:', items[['quantity', 'unit_price', 'line_total']].isna().sum().to_dict())
print('키 중복:', products.product_id.duplicated().sum(),
      orders.order_id.duplicated().sum(), items.order_item_id.duplicated().sum())

diff = (items.quantity * items.unit_price - items.line_total).abs()
print('line_total 불일치 행:', (diff > 0.01).sum(), '/', len(items))

print('category 원본 고유값:', products.category.nunique())
products['category'] = products.category.str.replace(' ', '', regex=False)
print('category 공백 제거 후:', products.category.nunique())

컬럼 존재: True True True
금액 결측: {'quantity': 0, 'unit_price': 0, 'line_total': 0}
키 중복: 0 0 0
line_total 불일치 행: 0 / 752
category 원본 고유값: 10
category 공백 제거 후: 7


In [6]:
m = items.merge(orders[['order_id', 'order_status']], on='order_id', how='left', indicator='m1')
print('items→orders 행 수:', len(items), '->', len(m), '/ 미매칭:', (m.m1 == 'left_only').sum())

m = m.drop(columns='m1').merge(products[['product_id', 'category']], on='product_id', how='left', indicator='m2')
print('→products 행 수:', len(m), '/ 미매칭:', (m.m2 == 'left_only').sum())

completed = m[m.order_status == 'completed']
matched = completed[completed.m2 == 'both']
print('completed 전체 합:', completed.line_total.sum())
print('카테고리 매칭된 합:', matched.line_total.sum())

group = matched.groupby('category').line_total.sum().sort_values(ascending=False)
print('그룹 합 == 매칭 합:', abs(group.sum() - matched.line_total.sum()) < 0.01)
display((group / group.sum() * 100).round(1))

items→orders 행 수: 752 -> 752 / 미매칭: 2
→products 행 수: 752 / 미매칭: 26
completed 전체 합: 141340000.0
카테고리 매칭된 합: 137325000.0
그룹 합 == 매칭 합: True


category
패션      24.0
식품      18.2
전자기기    15.7
뷰티      12.9
도서      12.9
스포츠      9.6
생활용품     6.6
Name: line_total, dtype: float64

### Evidence Matrix
| LLM 주장·가정 | 확인한 Evidence | 결과 | 판단 |
|---|---|---|---|
| 컬럼이 존재한다 | `in df.columns` | 모두 있음 | 사용 |
| 결측이 없다 | `isna().sum()` | 금액 관련 결측 0 | 사용 |
| 키가 고유하다 | `duplicated().sum()` | 중복 0 | 사용 |
| line_total = quantity × unit_price | 차이 계산 | 불일치 0 / 752 | 사용 |
| category 값이 깨끗하다 | `nunique()` | 10개 → 공백 제거 후 7개 | **수정 후 사용** |
| 병합이 정상이다 | 행 수, indicator | orders 2행, products 26행 미매칭 | **수정 후 사용** |
| 금액 합계가 맞다 | source total vs group total | 그룹 합 = 매칭 합 | 사용 |
| 모든 completed 금액이 반영된다 | completed 전체 합과 비교 | 4,015,000이 카테고리 불명으로 제외 | **보류(한계로 기록)** |

### 사람이 수정한 내용과 이유
- 카테고리 표기(`뷰 티` → `뷰티` 등)를 공백 제거로 통일했다. 안 하면 같은 카테고리가 둘로 갈라진다.
- 미매칭 행은 지우지 않고 제외 범위를 숫자로 기록했다.

### 한계
- 비중은 카테고리를 아는 completed 금액(137,325,000) 기준이다. 전체 completed 금액(141,340,000)의 약 97%다.
- 미매칭 원인(데이터 입력 오류인지, 삭제된 상품인지)은 현재 자료로는 알 수 없어 **가설**로만 남긴다.

## 7. LLM 사용 Log Template — 아직 실행하지 않음


In [7]:
usage_log = result['usage_log'].copy()
display(usage_log)
assert usage_log['execution_status'].eq('not_executed').all()
assert usage_log['final_use'].eq('not_used').all()


,step,execution_status,executed_at,provider,model,prompt_version,purpose,input_summary,response_summary,validation_result,revision_note,final_use
0,데이터 구조 설명,not_executed,,,,2.0,,,,,,not_used
1,분석 질문 생성,not_executed,,,,2.0,,,,,,not_used
2,전처리 계획,not_executed,,,,2.0,,,,,,not_used
3,시각화 설계,not_executed,,,,2.0,,,,,,not_used
4,회귀 코드 검토,not_executed,,,,2.0,,,,,,not_used
5,분류 코드 검토,not_executed,,,,2.0,,,,,,not_used
6,결과 해석,not_executed,,,,2.0,,,,,,not_used
7,외부 문서 검토,not_executed,,,,2.0,,,,,,not_used


`ch11_llm_usage_log.csv`는 파일명이 Log라고 되어 있어도 처음 생성될 때는 **빈 템플릿**입니다. 실제 LLM을 사용했다면 해당 행만 `execution_status=executed`로 바꾸고 `executed_at`, `provider`, `model`, `input_summary`, `response_summary`, `validation_result`, `revision_note`, `final_use`를 실제 기록으로 채웁니다.


### STEP 9 정리
- **결과 관찰:** 8개 단계 모두 `execution_status = not_executed`, `final_use = not_used`다. `provider`와 `model`은 비어 있다.
- **나의 해석과 판단:** 이번 실습에서는 외부 LLM에 Prompt를 실제로 넣지 않았다. 이 Log는 **실제 사용 증거가 아니라 빈 템플릿**이다.
- **사용 여부:** 실제 LLM 사용 없음
- **Evidence:** `assert`로 `not_executed`, `not_used` 확인
- **한계:** LLM 응답이 없으므로 STEP 8의 Evidence Matrix는 LLM 제안을 검증한 것이 아니라, STEP 1 질문에 필요한 가정을 직접 검증한 것이다.

In [ ]:
# 실제 사용 후 한 행을 수정하는 예시입니다. 아래 코드는 실행하지 않아도 됩니다.
# row = usage_log['step'].eq('분석 질문 생성')
# usage_log.loc[row, 'execution_status'] = 'executed'
# usage_log.loc[row, 'executed_at'] = '실제 실행 시각'
# usage_log.loc[row, 'provider'] = '실제 제공자'
# usage_log.loc[row, 'model'] = '실제 모델명'
# usage_log.loc[row, 'prompt_version'] = '2.0'
# usage_log.loc[row, 'input_summary'] = '개인정보 없는 safe context'
# usage_log.loc[row, 'response_summary'] = '실제 답변 요약'
# usage_log.loc[row, 'validation_result'] = '실제 검증 결과'
# usage_log.loc[row, 'revision_note'] = '사람이 수정한 내용'
# usage_log.loc[row, 'final_use'] = 'used / partial / not_used'


## 8. 생성 파일 확인


In [8]:
for name, path in result['output_paths'].items():
    print(name, 'OK' if path.exists() else 'MISSING', path)


dataset_summary OK C:\dev\llm-data-analysis-course\reports\ch11_dataset_summary_for_llm.csv
column_summary OK C:\dev\llm-data-analysis-course\reports\ch11_column_summary_for_llm.csv
sensitive_review OK C:\dev\llm-data-analysis-course\reports\ch11_sensitive_column_review.csv
safe_context OK C:\dev\llm-data-analysis-course\reports\ch11_safe_llm_context.md
context_validation OK C:\dev\llm-data-analysis-course\reports\ch11_safe_context_validation.csv
prompt_templates OK C:\dev\llm-data-analysis-course\reports\ch11_prompt_templates.csv
checklist OK C:\dev\llm-data-analysis-course\reports\ch11_llm_review_checklist.csv
usage_log OK C:\dev\llm-data-analysis-course\reports\ch11_llm_usage_log.csv
prompt_log OK C:\dev\llm-data-analysis-course\reports\ch11_llm_prompt_log.md


주요 Evidence에는 `ch11_safe_context_validation.csv`가 추가됩니다. Safe Context, Prompt Template, Checklist, Usage Log는 LLM의 답변 자체가 아니라 **안전한 입력 설계와 검증·기록을 위한 자료**입니다.


### STEP 10 정리
- **결과 관찰:** 생성 파일 9개가 모두 `OK`다. `source_type = processed`다.
- **확인한 것:** Safe Context Validation에 FAIL 없음, 실제 값 예시 자동 생성 없음, Prompt Template에 `human_review_required` 있음, Usage Log 초기 상태 `not_executed / not_used`, 외부 LLM 자동 호출 없음
- **Evidence:** `ch11_safe_context_validation.csv`(5개 PASS), `python scripts/run_llm_prompt_analysis.py` 재실행
- **한계:** 이 파일들은 LLM 답변이 아니라 안전한 입력 설계와 검증·기록을 위한 자료다.

## 9. 전체 자료 다시 생성하기

프로젝트 루트에서 다음 명령으로 같은 템플릿과 검증 자료를 다시 생성할 수 있습니다.

```powershell
python scripts/run_llm_prompt_analysis.py
```

이 명령은 외부 LLM을 호출하지 않습니다.


## 정리

Chapter 11의 핵심은 좋은 문장을 만드는 프롬프트 기술이 아니라 **안전한 Context → 검증 조건이 있는 Prompt → 답변 검증 → 사람 수정 → 실제 사용 기록**입니다. 다음 장에서는 LLM이 만든 분석 코드를 실행 전후에 더 구체적으로 검증합니다.
